In [73]:
import sqlite3

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import MessagesState

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db",
    check_same_thread=False,
)

config = {
    "configurable": {
        "thread_id": "0_x",
    },
}

In [74]:
class State(MessagesState):
    pass


graph_builder = StateGraph(State)

In [75]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],
    }

In [76]:
graph_builder.add_node("chatbot", chatbot)


graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)

In [79]:
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "나는 서울에 살고 있어",
            },
        ]
    },
    config=config,
)

In [80]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

안녕 내 이름은 김민기야 너는 누구야?
================================== Ai Message ==================================

안녕하세요, 김민기님! 저는 assistant, 여러분의 질문에 답하고 도움을 주기 위해 만들어진 인공지능입니다. 어떻게 도와드릴까요?
================================ Human Message =================================

나는 서울에 살고 있어
================================== Ai Message ==================================

서울에 살고 계시군요! 서울은 정말 멋진 도시죠. 어떤 곳이 가장 좋아하시나요? 또는 서울에서 즐기는 취미나 활동이 있나요?


In [81]:
state_history = graph.get_state_history(config)

In [82]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("=========\n")

()
[HumanMessage(content='안녕 내 이름은 김민기야 너는 누구야?', additional_kwargs={}, response_metadata={}, id='6f30b68b-ce37-48b3-99bb-c05c9d0dc280'), AIMessage(content='안녕하세요, 김민기님! 저는 assistant, 여러분의 질문에 답하고 도움을 주기 위해 만들어진 인공지능입니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 21, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_788bf556eb', 'id': 'chatcmpl-ECK27EG0fFVV9rZ9vzcBd2BVtadQM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ff9fe-b922-7590-994b-e63c14e0c860-0', usage_metadata={'input_tokens': 21, 'output_tokens': 36, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_

In [83]:
state_history = graph.get_state_history(config)

to_fork = list(state_history)[-5]

to_fork.config

{'configurable': {'thread_id': '0_x',
  'checkpoint_ns': '',
  'checkpoint_id': '1f196e91-d9d7-612e-8003-ed447414792e'}}

In [89]:
from langchain_core.messages import HumanMessage

graph.update_state(
    to_fork.config,
    {
        "messages": [
            HumanMessage(
                content="나는 강원도에 살고 있어",
                id="46a2ff5c-a5e9-45cf-ba8f-eaf30d1ba40b",
            )
        ]
    },
)

{'configurable': {'thread_id': '0_x',
  'checkpoint_ns': '',
  'checkpoint_id': '1f196eab-b2f6-6e14-8004-860c4223546e'}}

In [90]:
forked_state = graph.get_state_history(
    {
        "configurable": {
            "thread_id": "0_x",
            "checkpoint_ns": "",
            "checkpoint_id": "1f196eab-b2f6-6e14-8004-860c4223546e",
        }
    }
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='안녕 내 이름은 김민기야 너는 누구야?', additional_kwargs={}, response_metadata={}, id='6f30b68b-ce37-48b3-99bb-c05c9d0dc280'), AIMessage(content='안녕하세요, 김민기님! 저는 assistant, 여러분의 질문에 답하고 도움을 주기 위해 만들어진 인공지능입니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 21, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_788bf556eb', 'id': 'chatcmpl-ECK27EG0fFVV9rZ9vzcBd2BVtadQM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ff9fe-b922-7590-994b-e63c14e0c860-0', usage_metadata={'input_tokens': 21, 'output_tokens': 36, 'total_tokens': 57, 'input_token_details': {'audio': 0, 

In [91]:
result = graph.invoke(
    None,
    {
        "configurable": {
            "thread_id": "0_x",
            "checkpoint_ns": "",
            "checkpoint_id": "1f196eab-b2f6-6e14-8004-860c4223546e",
        }
    },
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

안녕 내 이름은 김민기야 너는 누구야?
================================== Ai Message ==================================

안녕하세요, 김민기님! 저는 assistant, 여러분의 질문에 답하고 도움을 주기 위해 만들어진 인공지능입니다. 어떻게 도와드릴까요?
================================ Human Message =================================

나는 강원도에 살고 있어
================================== Ai Message ==================================

강원도에 살고 계시군요! 강원도는 자연 경관이 아름답고, 사계절 내내 다양한 활동을 즐길 수 있는 곳이죠. 특히 겨울에는 스키나 눈놀이, 여름에는 해변이나 산행 같은 활동이 인기죠. 강원도에 대해 이야기하고 싶은 부분이 있나요? 혹시 궁금한 점이 있으면 말씀해 주세요!
